# 스트리밍 알고리즘 2종 구현 및 정확도·메모리 트레이드오프 분석

## 데이터셋

본 과제에서는 MovieLens 1M 데이터셋을 사용하였다.

* Dataset: MovieLens 1M
* Provider: GroupLens Research, University of Minnesota
* Dataset URL: https://files.grouplens.org/datasets/movielens/ml-1m.zip
* Official Page: https://grouplens.org/datasets/movielens/1m/

MovieLens 1M 데이터셋은 1,000,209개의 사용자-영화 평점 정보를 포함하고 있으며, 본 과제에서는 각 평점 기록을 데이터 스트림 이벤트로 간주하여 실험을 수행하였다.

## 구현 알고리즘

본 과제에서는 다음 두 가지 스트리밍 알고리즘을 구현하였다.

1. Bloom Filter
2. Count-Min Sketch

## 개발 환경

* Python 3
* Google Colab

## 생성형 AI 활용

본 과제의 수행 과정에서 생성형 AI(ChatGPT)를 활용하여 알고리즘 구현 방법, 실험 설계, 코드 작성, 결과 분석 및 보고서 작성 방향에 대한 참고를 받았다.

실험 실행, 결과 검증, 파라미터 설정 및 최종 제출물 구성은 직접 수행하였다.

## 1. 라이브러리 설치 및 import

In [ ]:
!pip install pympler -q

In [ ]:
import os
import zipfile
import time
import random
import hashlib
import math
import sys
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from pympler import asizeof

## 2. MovieLens 1M 다운로드 및 압축 해제

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip

In [ ]:
with zipfile.ZipFile("ml-1m.zip", "r") as zip_ref:
    zip_ref.extractall(".")

ratings_path = "ml-1m/ratings.dat"

print(os.listdir("ml-1m"))
print("ratings.dat exists:", os.path.exists(ratings_path))

## 3. 데이터 확인

In [ ]:
count = 0

with open(ratings_path, "r", encoding="latin-1") as f:
    for line in f:
        count += 1

print("총 평점 레코드 수:", count)

In [ ]:
with open(ratings_path, "r", encoding="latin-1") as f:
    for _ in range(5):
        print(f.readline().strip())

## 4. 스트림 제너레이터 작성

In [ ]:
def rating_stream(file_path):
    with open(file_path, "r", encoding="latin-1") as f:
        for line in f:
            user_id, movie_id, rating, timestamp = line.strip().split("::")
            yield {
                "user_id": user_id,
                "movie_id": movie_id,
                "rating": int(rating),
                "timestamp": int(timestamp),
                "event_id": f"{user_id}:{movie_id}"
            }

## 5. Bloom Filter 직접 구현

In [ ]:
class BloomFilter:
    def __init__(self, size, hash_count):
        self.size = size
        self.hash_count = hash_count
        self.bit_array = bytearray(size)

    def _hashes(self, item):
        item = str(item)
        for i in range(self.hash_count):
            data = f"{item}_{i}".encode("utf-8")
            digest = hashlib.md5(data).hexdigest()
            yield int(digest, 16) % self.size

    def add(self, item):
        for index in self._hashes(item):
            self.bit_array[index] = 1

    def contains(self, item):
        return all(self.bit_array[index] == 1 for index in self._hashes(item))

    def memory_usage(self):
        return asizeof.asizeof(self.bit_array)

## 6. Count-Min Sketch 직접 구현

In [ ]:
class CountMinSketch:
    def __init__(self, width, depth):
        self.width = width
        self.depth = depth
        self.table = [[0] * width for _ in range(depth)]

    def _hashes(self, item):
        item = str(item)
        for i in range(self.depth):
            data = f"{item}_{i}".encode("utf-8")
            digest = hashlib.md5(data).hexdigest()
            yield int(digest, 16) % self.width

    def add(self, item, count=1):
        for row, col in enumerate(self._hashes(item)):
            self.table[row][col] += count

    def estimate(self, item):
        return min(self.table[row][col] for row, col in enumerate(self._hashes(item)))

    def memory_usage(self):
        return asizeof.asizeof(self.table)

## 7. Ground Truth 계산

In [ ]:
true_events = set()
true_movie_counts = defaultdict(int)

start_time = time.time()

for record in rating_stream(ratings_path):
    event_id = record["event_id"]
    movie_id = record["movie_id"]

    true_events.add(event_id)
    true_movie_counts[movie_id] += 1

ground_truth_time = time.time() - start_time

print("Ground Truth 생성 완료")
print("고유 event 수:", len(true_events))
print("고유 movie 수:", len(true_movie_counts))
print("처리 시간:", ground_truth_time)
print("set 메모리:", asizeof.asizeof(true_events), "bytes")
print("dict 메모리:", asizeof.asizeof(true_movie_counts), "bytes")

## 8. Bloom Filter 실험 함수

In [ ]:
def run_bloom_filter_experiment(size, hash_count):
    bf = BloomFilter(size=size, hash_count=hash_count)

    start_time = time.time()

    for record in rating_stream(ratings_path):
        bf.add(record["event_id"])

    elapsed_time = time.time() - start_time

    # False Positive 측정용: 실제로 존재하지 않는 event_id 생성
    test_size = 10000
    false_positive = 0

    for i in range(test_size):
        fake_event = f"fake_user_{i}:fake_movie_{i}"

        if fake_event in true_events:
            continue

        if bf.contains(fake_event):
            false_positive += 1

    false_positive_rate = false_positive / test_size

    return {
        "algorithm": "Bloom Filter",
        "size": size,
        "hash_count": hash_count,
        "false_positive_rate": false_positive_rate,
        "memory_bytes": bf.memory_usage(),
        "time_sec": elapsed_time,
        "throughput_records_per_sec": count / elapsed_time
    }

## 9. Count-Min Sketch 실험 함수

In [ ]:
def run_cms_experiment(width, depth):
    cms = CountMinSketch(width=width, depth=depth)

    start_time = time.time()

    for record in rating_stream(ratings_path):
        cms.add(record["movie_id"])

    elapsed_time = time.time() - start_time

    absolute_errors = []
    relative_errors = []

    for movie_id, true_count in true_movie_counts.items():
        estimated_count = cms.estimate(movie_id)
        abs_error = estimated_count - true_count
        rel_error = abs_error / true_count

        absolute_errors.append(abs_error)
        relative_errors.append(rel_error)

    avg_absolute_error = sum(absolute_errors) / len(absolute_errors)
    avg_relative_error = sum(relative_errors) / len(relative_errors)
    max_absolute_error = max(absolute_errors)

    return {
        "algorithm": "Count-Min Sketch",
        "width": width,
        "depth": depth,
        "avg_absolute_error": avg_absolute_error,
        "avg_relative_error": avg_relative_error,
        "max_absolute_error": max_absolute_error,
        "memory_bytes": cms.memory_usage(),
        "time_sec": elapsed_time,
        "throughput_records_per_sec": count / elapsed_time
    }

## 10. 파라미터 비교 실험 실행

### Bloom Filter

In [ ]:
bloom_params = [
    (500000, 3),
    (1000000, 3),
    (2000000, 3),
    (1000000, 5),
    (2000000, 5)
]

bloom_results = []

for size, hash_count in bloom_params:
    result = run_bloom_filter_experiment(size, hash_count)
    bloom_results.append(result)
    print(result)

In [ ]:
bloom_df = pd.DataFrame(bloom_results)
bloom_df

### Count-Min Sketch

In [ ]:
cms_params = [
    (500, 3),
    (1000, 3),
    (2000, 3),
    (1000, 5),
    (2000, 5)
]

cms_results = []

for width, depth in cms_params:
    result = run_cms_experiment(width, depth)
    cms_results.append(result)
    print(result)

In [ ]:
cms_df = pd.DataFrame(cms_results)
cms_df

## 11. 결과 표 저장

In [ ]:
bloom_df.to_csv("bloom_filter_results.csv", index=False)
cms_df.to_csv("count_min_sketch_results.csv", index=False)

print("결과 CSV 저장 완료")

## 12. Bloom Filter 결과 시각화

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(bloom_df["memory_bytes"], bloom_df["false_positive_rate"], marker="o")
plt.xlabel("Memory Usage (bytes)")
plt.ylabel("False Positive Rate")
plt.title("Bloom Filter: Memory vs False Positive Rate")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(bloom_df["memory_bytes"], bloom_df["time_sec"], marker="o")
plt.xlabel("Memory Usage (bytes)")
plt.ylabel("Processing Time (sec)")
plt.title("Bloom Filter: Memory vs Processing Time")
plt.grid(True)
plt.show()

## 13. Count-Min Sketch 결과 시각화

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cms_df["memory_bytes"], cms_df["avg_relative_error"], marker="o")
plt.xlabel("Memory Usage (bytes)")
plt.ylabel("Average Relative Error")
plt.title("Count-Min Sketch: Memory vs Average Relative Error")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cms_df["memory_bytes"], cms_df["time_sec"], marker="o")
plt.xlabel("Memory Usage (bytes)")
plt.ylabel("Processing Time (sec)")
plt.title("Count-Min Sketch: Memory vs Processing Time")
plt.grid(True)
plt.show()

## 14. 최종 결과 합치기

In [ ]:
bloom_summary = bloom_df.copy()
cms_summary = cms_df.copy()

display(bloom_summary)
display(cms_summary)